# LazyAd — Phase 1, Spike S2: Nano Banana Hero Image Generation

**Source of Truth:** `Docs/Phase 1, Spike S2.md` & `Docs/PHASE 0 FEASIBILITY & REQUIREMENTS.md` (Approved v2.2)

### Objective
Empirically verify that **Nano Banana** (`gemini-3.1-flash-image`), accessed through the Google Gemini API, can generate and save a high-quality luxury watch hero image optimized specifically for single-image 3D reconstruction (TripoSR in Spike S3).

### Key Model Details & API Caveats
- **Model Candidate:** `gemini-3.1-flash-image` (Nano Banana 2), Google's native multimodal image generation model.
- **SDK:** `google-genai` Python SDK (`client.models.generate_content`).
- **Quota & Billing:** Free-tier or existing Google AI API quota applies. If your API key lacks access or triggers quota limits, the notebook reports the exact API response without crashing.
- **Security:** API keys are prompted securely via `getpass.getpass()`. Keys are never saved, printed, or written to disk.

> **Strict Scope Limits:** Spike S2 is isolated to hero image generation and verification. It does **not** install 3D models (TripoSR), does **not** generate GLBs, and does **not** render video.

## Step 1: Environment Check & SDK Upgrade
Verifies the installed `google-genai` SDK and ensures Pillow is available.

In [ ]:
!pip install --upgrade -q google-genai pillow

import google.genai as genai
import PIL
print(f"google-genai SDK version : {getattr(genai, '__version__', 'Installed')}")
print(f"Pillow (PIL) version     : {PIL.__version__}")

## Step 2: Secure API Key Input & Client Initialization
Enter your Gemini API key securely. The key will **never** be printed or saved to any file.

In [ ]:
import os
import getpass
from google import genai
from google.genai import types

# Check if already set in environment; otherwise prompt securely
api_key = os.environ.get("GEMINI_API_KEY", None)
if not api_key:
    api_key = getpass.getpass("Enter your Gemini API Key (input hidden): ").strip()

if not api_key:
    raise ValueError("API key is required to execute Spike S2.")

# Initialize GenAI client
client = genai.Client(api_key=api_key)
MODEL_ID = "gemini-3.1-flash-image"

print(f"✅ GenAI Client initialized successfully for model: {MODEL_ID}")
print("🔒 API Key stored securely in transient runtime memory (not logged).")

## Step 3: Fictional Luxury Watch Prompt Configuration
The prompt is structured with strict constraints optimized for **single-image 3D reconstruction**:
- One complete wristwatch, three-quarter front angle (dial, case, bezel, and strap visible).
- Centered product, occupying most of the frame.
- Solid, uniform light-grey studio cyclorama background with clear silhouette separation.
- Negative constraints: No human hands, no extra objects, no legible brand text or logos, no extreme glare or deep shadows.

In [ ]:
# Deterministic, editable product facts template
PRODUCT_FACTS = {
    "brand": "Aethelgard (Fictional Luxury Atelier)",
    "model_name": "Chrono Minimaliste",
    "case_material": "Brushed black titanium with chamfered polished lugs",
    "bezel": "Slim circular bezel with discrete geometric index notches",
    "dial_color": "Deep obsidian sunburst black dial",
    "hands": "Brushed silver skeleton baton hands",
    "indices": "Minimalist polished silver applied hour batons, no typography, no brand text",
    "strap": "Matte textured black alligator leather strap with tone-on-tone stitching",
    "crown": "Fluted titanium cylindrical winding crown at 3 o'clock position"
}

# Construct prompt optimized specifically for 3D reconstruction
HERO_PROMPT = (
    f"High-end commercial studio product photography of one complete wristwatch: {PRODUCT_FACTS['model_name']}. "
    f"The watch features a {PRODUCT_FACTS['case_material']}, a {PRODUCT_FACTS['bezel']}, "
    f"an {PRODUCT_FACTS['dial_color']} with {PRODUCT_FACTS['indices']} and {PRODUCT_FACTS['hands']}, "
    f"a {PRODUCT_FACTS['crown']}, and a {PRODUCT_FACTS['strap']}. "
    "Camera angle: 45-degree three-quarter front perspective view showing the full watch face, side case profile, and strap curvature. "
    "Framing: Perfectly centered product, occupying 80% of the frame, isolated in full. "
    "Lighting: Soft diffused multi-source studio softbox lighting with subtle realistic metallic reflections along the case edges. "
    "Background: Completely solid, plain neutral light-grey studio cyclorama background with razor-sharp silhouette separation and no background clutter. "
    "Strict negative constraints: Absolutely no people, no human hands, no wrist, no watch box, no packaging, "
    "no secondary decorative props, no legible brand text, no logos, no watermarks, no harsh glare, no extreme clipping shadows."
)

print("=" * 72)
print("LAZYAD — S2 HERO IMAGE PROMPT TEMPLATE")
print("=" * 72)
print(HERO_PROMPT)
print("=" * 72)

## Step 4: Execute API Generation Call (Nano Banana / `gemini-3.1-flash-image`)

In [ ]:
import time
import io
from PIL import Image

# Destination asset directory
ASSET_DIR = "/content/lazyad_assets"
os.makedirs(ASSET_DIR, exist_ok=True)
HERO_IMAGE_PATH = os.path.join(ASSET_DIR, "hero_watch.png")

print(f"Submitting generation request to model '{MODEL_ID}'...")
start_time = time.time()

response = None
api_error = None
raw_image = None

try:
    # Gemini 3.1 Flash Image uses generate_content with IMAGE + TEXT response modalities
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=HERO_PROMPT,
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE", "TEXT"],
            image_config=types.ImageConfig(
                aspect_ratio="1:1",
                output_mime_type="image/png",
            ),
        ),
    )
except Exception as e:
    api_error = str(e)
    print(f"\n❌ API Call Failed: {api_error}")

duration = time.time() - start_time
print(f"Request completed in: {duration:.2f} seconds")

## Step 5: Extract, Validate, and Save Hero Image

In [ ]:
image_extracted = False
model_notes = []

if response is not None:
    # Gather all candidate parts safely
    parts = getattr(response, "parts", None)
    if not parts and hasattr(response, "candidates") and response.candidates:
        parts = getattr(response.candidates[0].content, "parts", [])
        
    if parts:
        for part in parts:
            if getattr(part, "text", None):
                model_notes.append(part.text)
            elif getattr(part, "inline_data", None):
                # Extract image using SDK helper or raw bytes
                if hasattr(part, "as_image") and callable(part.as_image):
                    raw_image = part.as_image()
                elif hasattr(part.inline_data, "data"):
                    raw_image = Image.open(io.BytesIO(part.inline_data.data))
                
                if raw_image is not None:
                    raw_image.save(HERO_IMAGE_PATH, format="PNG")
                    image_extracted = True
                    break

# File verification checks
file_exists = os.path.exists(HERO_IMAGE_PATH)
file_size_kb = 0.0
img_dimensions = (0, 0)
img_mode = "N/A"
verified_openable = False

if file_exists:
    file_size_bytes = os.path.getsize(HERO_IMAGE_PATH)
    file_size_kb = file_size_bytes / 1024.0
    if file_size_bytes > 0:
        try:
            with Image.open(HERO_IMAGE_PATH) as test_img:
                test_img.verify()
            with Image.open(HERO_IMAGE_PATH) as test_img:
                img_dimensions = test_img.size
                img_mode = test_img.mode
                verified_openable = True
        except Exception as img_err:
            print(f"⚠️ Image verification error: {img_err}")

print("=" * 72)
print("LAZYAD — SPIKE S2 IMAGE EXTRACTION & VALIDATION REPORT")
print("=" * 72)
print(f"• Target Output Path   : {HERO_IMAGE_PATH}")
print(f"• Image Extracted      : {image_extracted}")
print(f"• File Exists on Disk  : {file_exists}")
print(f"• File Size            : {file_size_kb:.2f} KB")
print(f"• Dimensions (WxH)     : {img_dimensions[0]} x {img_dimensions[1]} pixels")
print(f"• Color Mode           : {img_mode}")
print(f"• Verified Openable    : {verified_openable}")
if model_notes:
    print(f"• Model Note Output    : {' '.join(model_notes)}")
print("=" * 72)

## Step 6: Inline Image Display & Visual Inspection

In [ ]:
from IPython.display import display, Image as IPImage

if verified_openable:
    print("\nGenerated Hero Image (Saved to disk):")
    display(IPImage(filename=HERO_IMAGE_PATH))
else:
    print("❌ No valid image to display. Review the API response above.")

## Step 7: Evaluated Pass / Fail Verdict (S2 Acceptance Criteria)

### Pass Criteria (from `Docs/Phase 1, Spike S2.md`)
1. Selected Nano Banana model is successfully called through the Gemini API.
2. A valid, non-empty image is returned.
3. Image is saved to `/content/lazyad_assets/hero_watch.png`.
4. Saved image can be reopened and displayed.
5. Image is suitable for single-image 3D reconstruction (centered, plain background, three-quarter view, no people/props).
6. Any API errors or quota restrictions are clearly documented.

In [ ]:
print("=" * 72)
print("SPIKE S2: EVALUATION & ACCEPTANCE VERDICT")
print("=" * 72)

c1_api_success = (api_error is None) and (response is not None)
c2_image_valid = image_extracted and verified_openable and (file_size_kb > 10.0)
c3_saved_correctly = file_exists and (HERO_IMAGE_PATH == "/content/lazyad_assets/hero_watch.png")
c4_dimensions_sane = (img_dimensions[0] >= 512) and (img_dimensions[1] >= 512)

print(f"1. API Call Succeeded              : [{'PASS' if c1_api_success else 'FAIL'}]")
print(f"2. Valid Image Data Returned       : [{'PASS' if c2_image_valid else 'FAIL'}]")
print(f"3. Saved to Target Path on Disk    : [{'PASS' if c3_saved_correctly else 'FAIL'}]")
print(f"4. Resolution Sufficient (>=512x512): [{'PASS' if c4_dimensions_sane else 'FAIL'}] ({img_dimensions[0]}x{img_dimensions[1]})")

technical_pass = c1_api_success and c2_image_valid and c3_saved_correctly and c4_dimensions_sane

print("-" * 72)
if technical_pass:
    print("🎯 TECHNICAL VERDICT: [PASS]")
    print(f"   Model: {MODEL_ID}")
    print(f"   Saved: {HERO_IMAGE_PATH} ({file_size_kb:.1f} KB, {img_dimensions[0]}x{img_dimensions[1]})")
    print(f"   Generation Time: {duration:.2f}s")
    print("\n🔎 VISUAL INSPECTION CHECKLIST FOR 3D RECONSTRUCTION:")
    print("   [ ] Is the watch centered and occupying most of the frame?")
    print("   [ ] Is the background plain and free of clutter / gradients?")
    print("   [ ] Are the dial, bezel, case profile, and strap clearly separated?")
    print("   [ ] Are there ZERO human hands, people, or secondary objects?")
    print("\n👉 Please inspect the displayed image and share the result to confirm S2 Pass!")
else:
    print("❌ TECHNICAL VERDICT: [FAIL]")
    if api_error:
        print(f"   Failure Reason: API Error -> {api_error}")
        if "429" in api_error or "RESOURCE_EXHAUSTED" in api_error:
            print("   Action: Quota exceeded on this API key. Wait for quota refresh or use an account with available quota.")
        elif "403" in api_error or "PERMISSION_DENIED" in api_error or "NOT_FOUND" in api_error:
            print(f"   Action: Model '{MODEL_ID}' may not be enabled on this key tier or requires billing.")
    elif not file_exists:
        print("   Failure Reason: Output file was not created.")
print("=" * 72)
